# Part 8 · ECoRe innovations 1–3

This notebook freezes the Part 7 encoder and tests three claims on source-heldout evidence: source-balanced author distributions, environment-relative geometry, and author-heldout episodic transfer. It does not rebuild Part 7 embeddings and does not train a production encoder.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
EXP = REPO / 'artifacts/source_expansion_v2'
HELDOUT = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
EMBEDDINGS = EXP / 'expanded_source_heldout_eval'
OUT = EXP / 'ecore_innovations_v1'
assert HELDOUT.exists(), f'Run Parts 1–7 first; missing {HELDOUT}'
for name in ('style_embedding_train_embeddings.npy', 'style_embedding_eval_embeddings.npy', 'style_embedding_scores.npz'):
    assert (EMBEDDINGS / name).exists(), f'Part 7 embedding artifact missing: {EMBEDDINGS / name}'
assert (REPO / 'scripts/evaluate_ecore_innovations.py').exists(), 'Pull the commit containing the redesigned Part 8 script'

def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')

# Remove only interrupted legacy Part 8 model directories. Completed checkpoints are preserved.
legacy_root = EXP / 'top4_full_retrain'
if legacy_root.exists():
    for candidate in legacy_root.glob('*/finetuned_authorship_expanded'):
        if not (candidate / 'training_config.json').exists():
            print('REMOVE interrupted legacy checkpoint:', candidate)
            shutil.rmtree(candidate)
    for model_name in ('strict_eval_finetuned_authorship', 'production_full_source_finetuned_authorship'):
        for candidate in legacy_root.glob(f'*/{model_name}'):
            if not (candidate / 'training_config.json').exists():
                print('REMOVE interrupted checkpoint:', candidate)
                shutil.rmtree(candidate)

print('Frozen input:', HELDOUT)
print('Reusing Part 7 embeddings:', EMBEDDINGS)
print('No encoder batches or production retraining will run in Part 8.')

## 8.1–8.3 · Falsifiable tests

Innovation 1 compares one centroid, hard prototype, and source-balanced soft prototypes. Innovation 2 uses language × register cohort-centred residual geometry and a shuffled-environment control. Innovation 3 trains a candidate-identity-free pairwise scorer on variable support episodes and evaluates it by whole-author cross-fitting.

In [ ]:
run([
    sys.executable, 'scripts/evaluate_ecore_innovations.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(OUT),
    '--temperature', '0.08',
    '--author-folds', '5',
    '--hard-negatives', '12',
    '--bootstrap-runs', '5000',
    '--train-cap', '300',
    '--embedding-seed', '20260701',
    '--seed', '20260725',
])

report_path = OUT / 'ecore_innovation_metrics.json'
report = json.loads(report_path.read_text())
summary = pd.DataFrame([
    {
        'innovation': 1,
        'claim': 'author as a source-balanced distribution',
        **report['innovation_1_distributional_profile']['paired_profile_bootstrap'],
        'supported': report['innovation_1_distributional_profile']['supported'],
    },
    {
        'innovation': 2,
        'claim': 'environment-relative evidence',
        **report['innovation_2_cohort_relative_geometry']['paired_profile_bootstrap'],
        'supported': report['innovation_2_cohort_relative_geometry']['supported'],
    },
    {
        'innovation': 3,
        'claim': 'whole-author episodic transfer',
        **report['innovation_3_episodic_transfer']['paired_profile_bootstrap'],
        'supported': report['innovation_3_episodic_transfer']['supported'],
    },
])
display(summary)
display(pd.DataFrame(report['test_metrics']).T.sort_values('mrr', ascending=False))
display(pd.DataFrame(report['innovation_3_episodic_transfer']['variable_support_test_metrics']).T)
print('RETURN:', report_path)
print('RETURN:', OUT / 'ecore_innovation_scores.npz')

## Decision rule

A claim passes only when its paired author-profile bootstrap 95% interval is wholly above zero. Innovation 2 must additionally beat the shuffled-environment control. Unsupported claims remain negative results; Part 8 does not hide them by training a larger encoder or adding more indicators.